# A1 Bear Camera AI - Colab GPU Training

Use this notebook from VSCode with the Google Colab extension or directly in Colab.

Runtime checklist:

1. Select a GPU runtime, such as T4, before training.
2. Run all cells. By default, the notebook downloads public COCO bear data inside Colab, so no local dataset upload is required.
3. Download `a1_camera_ai_colab_artifacts.zip` and extract it at the repository root on the PC or Raspberry Pi.

Safety boundary: this camera AI is only an additional perception signal. It does not replace the Arduino Uno Q contact-pad state machine or fail-safe `RELEASE_OFF` behavior.


In [5]:
from pathlib import Path
import os
import shutil
import subprocess
import sys
import zipfile

print(sys.version)
try:
    subprocess.run(["nvidia-smi"], check=False)
except FileNotFoundError:
    print("nvidia-smi was not found. Check that the Colab runtime type is GPU.")


3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]


In [6]:
%pip install -q ultralytics==8.3.40 opencv-python==4.10.0.84 PyYAML==6.0.2


In [7]:
import torch
import ultralytics

print("ultralytics", ultralytics.__version__)
print("torch", torch.__version__)
print("cuda_available", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("GPU is not available. In Colab, choose Runtime > Change runtime type > GPU, then reconnect.")


ultralytics 8.3.40
torch 2.11.0+cu128
cuda_available True
gpu Tesla T4


## Dataset input

Default flow: download COCO 2017 `bear` annotations and images directly inside Colab.

Fallback flows are still available: set `DATASET_MODE` to `upload_zip` or `google_drive_zip` if you want to use `data/packages/bear_public_yolo_pi.zip` instead.


In [8]:
DATASET_MODE = "auto_coco"  # auto_coco, upload_zip, or google_drive_zip
COCO_MAX_IMAGES = 80
COCO_VALIDATION_FRACTION = 0.2
COCO_RANDOM_SEED = 7
DRIVE_DATASET_ZIP = "/content/drive/MyDrive/a1_bear_honey/bear_public_yolo_pi.zip"
UPLOAD_DATASET_ZIP_NAME = "bear_public_yolo_pi.zip"

workspace = Path("/content/2026_Hackathon")
if workspace.exists():
    shutil.rmtree(workspace)
workspace.mkdir(parents=True, exist_ok=True)
dataset_dir = workspace / "data" / "datasets" / "bear_public_yolo"

if DATASET_MODE == "google_drive_zip":
    from google.colab import drive
    drive.mount("/content/drive")
    dataset_zip = Path(DRIVE_DATASET_ZIP)
elif DATASET_MODE == "upload_zip":
    from google.colab import files
    dataset_zip = Path("/content") / UPLOAD_DATASET_ZIP_NAME
    if not dataset_zip.exists():
        print(f"Upload {UPLOAD_DATASET_ZIP_NAME}")
        uploaded = files.upload()
        if UPLOAD_DATASET_ZIP_NAME not in uploaded:
            raise RuntimeError(f"Expected upload file: {UPLOAD_DATASET_ZIP_NAME}")
elif DATASET_MODE == "auto_coco":
    dataset_zip = None
else:
    raise ValueError(f"Unsupported DATASET_MODE: {DATASET_MODE}")

if dataset_zip is not None and not dataset_zip.exists():
    raise FileNotFoundError(dataset_zip)
print("DATASET_MODE", DATASET_MODE)
if dataset_zip is not None:
    print("dataset_zip", dataset_zip, dataset_zip.stat().st_size)


DATASET_MODE auto_coco


In [9]:
import json
import random
import urllib.request

COCO_ANNOTATIONS_URL = "http://images.cocodataset.org/annotations/annotations_trainval2017.zip"
COCO_IMAGE_URL_TEMPLATE = "http://images.cocodataset.org/val2017/{file_name}"

def download_file(url, output_path):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    if output_path.exists() and output_path.stat().st_size > 0:
        return output_path
    print("Downloading", url)
    urllib.request.urlretrieve(url, output_path)
    return output_path

def ensure_coco_val_annotations(download_dir):
    download_dir = Path(download_dir)
    zip_path = download_dir / "annotations_trainval2017.zip"
    annotations_path = download_dir / "annotations" / "instances_val2017.json"
    if annotations_path.exists():
        return annotations_path
    download_file(COCO_ANNOTATIONS_URL, zip_path)
    with zipfile.ZipFile(zip_path) as archive:
        archive.extract("annotations/instances_val2017.json", download_dir)
    return annotations_path

def coco_bbox_to_yolo(bbox_xywh, image_width, image_height):
    x_min, y_min, width, height = [float(value) for value in bbox_xywh]
    return (
        max(0.0, min(1.0, (x_min + width / 2.0) / float(image_width))),
        max(0.0, min(1.0, (y_min + height / 2.0) / float(image_height))),
        max(0.0, min(1.0, width / float(image_width))),
        max(0.0, min(1.0, height / float(image_height))),
    )

def build_coco_yolo_dataset(dataset_dir, max_images, validation_fraction, seed):
    dataset_dir = Path(dataset_dir)
    annotations_path = ensure_coco_val_annotations(Path("/content/coco_cache"))
    coco = json.loads(annotations_path.read_text(encoding="utf-8"))
    bear_category_id = next(category["id"] for category in coco["categories"] if category["name"] == "bear")
    images_by_id = {int(image["id"]): image for image in coco["images"]}
    annotations_by_image_id = {}
    for annotation in coco["annotations"]:
        if int(annotation["category_id"]) == int(bear_category_id):
            annotations_by_image_id.setdefault(int(annotation["image_id"]), []).append(annotation)

    image_ids = sorted(annotations_by_image_id)
    random.Random(seed).shuffle(image_ids)
    selected_image_ids = image_ids[:max_images]
    val_count = max(1, int(len(selected_image_ids) * validation_fraction))
    val_ids = set(selected_image_ids[:val_count])

    for image_id in selected_image_ids:
        split_name = "val" if image_id in val_ids else "train"
        image = images_by_id[image_id]
        image_file = str(image["file_name"])
        image_path = dataset_dir / "images" / split_name / image_file
        label_path = dataset_dir / "labels" / split_name / f"{Path(image_file).stem}.txt"
        download_file(COCO_IMAGE_URL_TEMPLATE.format(file_name=image_file), image_path)
        label_path.parent.mkdir(parents=True, exist_ok=True)
        label_lines = []
        for annotation in annotations_by_image_id[image_id]:
            x_center, y_center, width, height = coco_bbox_to_yolo(
                annotation["bbox"], image["width"], image["height"]
            )
            label_lines.append(f"0 {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}")
        label_path.write_text("\n".join(label_lines) + "\n", encoding="utf-8")

if DATASET_MODE == "auto_coco":
    build_coco_yolo_dataset(
        dataset_dir,
        max_images=COCO_MAX_IMAGES,
        validation_fraction=COCO_VALIDATION_FRACTION,
        seed=COCO_RANDOM_SEED,
    )
else:
    with zipfile.ZipFile(dataset_zip) as archive:
        archive.extractall(workspace)

train_image_count = len(list((dataset_dir / "images" / "train").glob("*.jpg")))
val_image_count = len(list((dataset_dir / "images" / "val").glob("*.jpg")))
print("dataset_dir", dataset_dir)
print("train_images", train_image_count)
print("val_images", val_image_count)

if train_image_count == 0 or val_image_count == 0:
    raise RuntimeError("Dataset split is empty. Check DATASET_MODE or recreate bear_public_yolo_pi.zip on the PC.")

colab_dataset_yaml = dataset_dir / "dataset_colab.yaml"
colab_dataset_yaml.write_text(
    "\n".join([
        f"path: {dataset_dir.as_posix()}",
        "train: images/train",
        "val: images/val",
        "names:",
        "  0: bear",
        "",
    ]),
    encoding="utf-8",
)
print(colab_dataset_yaml.read_text())


dataset_dir /content/2026_Hackathon/data/datasets/bear_public_yolo
train_images 40
val_images 9
path: /content/2026_Hackathon/data/datasets/bear_public_yolo
train: images/train
val: images/val
names:
  0: bear



## Train

The defaults are intentionally small for free Colab GPUs. Increase `EPOCHS` after the first successful run.


In [10]:
from ultralytics import YOLO

BASE_MODEL = "yolov8n.pt"
EPOCHS = 50
IMGSZ = 256
BATCH = 16
WORKERS = 2

model = YOLO(BASE_MODEL)
results = model.train(
    data=str(colab_dataset_yaml),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=0,
    workers=WORKERS,
    project="/content/runs/camera_ai_train",
    name="bear_yolo_colab",
    exist_ok=True,
    cache=True,
)

save_dir = Path(results.save_dir)
best_model = save_dir / "weights" / "best.pt"
print("best_model", best_model)
if not best_model.exists():
    raise FileNotFoundError(best_model)


100%|██████████| 6.25M/6.25M [00:00<00:00, 105MB/s]

New https://pypi.org/project/ultralytics/8.4.66 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.40 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)


engine/trainer: task=detect, mode=train, model=yolov8n.pt, data=/content/2026_Hackathon/data/datasets/bear_public_yolo/dataset_colab.yaml, epochs=50, time=None, patience=100, batch=16, imgsz=256, save=True, save_period=-1, cache=True, device=0, workers=2, project=/content/runs/camera_ai_train, name=bear_yolo_colab, exist_ok=True, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, embed=None, show=False, save_frames=False, save_txt=False, save_conf=False, save_crop=False, show_labels=True, show_conf=True, show_boxes=True

100%|██████████| 755k/755k [00:00<00:00, 24.6MB/s]


Overriding model.yaml nc=80 with nc=1

                   from  n    params  module                                       arguments                     
  0                  -1  1       464  ultralytics.nn.modules.conv.Conv             [3, 16, 3, 2]                 
  1                  -1  1      4672  ultralytics.nn.modules.conv.Conv             [16, 32, 3, 2]                
  2                  -1  1      7360  ultralytics.nn.modules.block.C2f             [32, 32, 1, True]             
  3                  -1  1     18560  ultralytics.nn.modules.conv.Conv             [32, 64, 3, 2]                
  4                  -1  2     49664  ultralytics.nn.modules.block.C2f             [64, 64, 2, True]             
  5                  -1  1     73984  ultralytics.nn.modules.conv.Conv             [64, 128, 3, 2]               
  6                  -1  2    197632  ultralytics.nn.modules.block.C2f             [128, 128, 2, True]           
  7                  -1  1    295424  ultralytics

...

      50/50     0.409G      0.489     0.6641     0.8328         10        256: 100%|██████████| 3/3 [00:00<00:00, 11.52it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 1/1 [00:00<00:00, 19.66it/s]

                   all          9         14      0.454      0.643      0.551      0.383



50 epochs completed in 0.015 hours.
Optimizer stripped from /content/runs/camera_ai_train/bear_yolo_colab/weights/last.pt, 6.2MB
Optimizer stripped from /content/runs/camera_ai_train/bear_yolo_colab/weights/best.pt, 6.2MB

Validating /content/runs/camera_ai_train/bear_yolo_colab/weights/best.pt...
Ultralytics 8.3.40 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 168 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 1/1 [00:00<00:00, 18.22it/s]


                   all          9         14      0.857      0.643      0.674      0.456
Speed: 0.1ms preprocess, 1.7ms inference, 0.0ms loss, 0.9ms postprocess per image
Results saved to /content/runs/camera_ai_train/bear_yolo_colab
best_model /content/runs/camera_ai_train/bear_yolo_colab/weights/best.pt


## Export for Raspberry Pi

NCNN is preferred for the Raspberry Pi runtime. If NCNN export fails in the free Colab environment, the notebook creates an ONNX fallback.


In [11]:
models_dir = workspace / "models"
models_dir.mkdir(parents=True, exist_ok=True)
runtime_pt = models_dir / "yolo_bear.pt"
shutil.copy2(best_model, runtime_pt)
print("runtime_pt", runtime_pt)

def replace_path(source, destination):
    source = Path(source)
    destination = Path(destination)
    if destination.exists():
        if destination.is_dir():
            shutil.rmtree(destination)
        else:
            destination.unlink()
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.move(str(source), str(destination))

exported_model_path = None
try:
    ncnn_exported = Path(YOLO(str(runtime_pt)).export(format="ncnn", imgsz=IMGSZ))
    ncnn_runtime = models_dir / "yolo_bear_ncnn_model"
    if ncnn_exported.resolve() != ncnn_runtime.resolve():
        replace_path(ncnn_exported, ncnn_runtime)
    exported_model_path = ncnn_runtime
    print("ncnn_runtime", ncnn_runtime)
except Exception as exc:
    print("NCNN export failed, creating ONNX fallback:", repr(exc))
    onnx_exported = Path(YOLO(str(runtime_pt)).export(format="onnx", imgsz=IMGSZ))
    onnx_runtime = models_dir / "yolo_bear.onnx"
    if onnx_exported.resolve() != onnx_runtime.resolve():
        replace_path(onnx_exported, onnx_runtime)
    exported_model_path = onnx_runtime
    print("onnx_runtime", onnx_runtime)

print("exported_model_path", exported_model_path)


runtime_pt /content/2026_Hackathon/models/yolo_bear.pt
Ultralytics 8.3.40 🚀 Python-3.12.13 torch-2.11.0+cu128 CPU (Intel Xeon 2.00GHz)
Model summary (fused): 168 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs

PyTorch: starting from '/content/2026_Hackathon/models/yolo_bear.pt' with input shape (1, 3, 256, 256) BCHW and output shape(s) (1, 5, 1344) (5.9 MB)

TorchScript: starting export with torch 2.11.0+cu128...
TorchScript: export success ✅ 1.2s, saved as '/content/2026_Hackathon/models/yolo_bear.torchscript' (11.8 MB)
requirements: Ultralytics requirement ['ncnn'] not found, attempting AutoUpdate...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 71.0 MB/s eta 0:00:00

requirements: AutoUpdate success ✅ 3.5s, installed 1 package: ['ncnn']
requirements: ⚠️ Restart runtime or rerun command for updates to take effect


NCNN: starting export with NCNN 1.0.20260526...
NCNN: WARNING ⚠️ PNNX not found. Attempting to download binary file from https://github.com/pnnx/pnnx/.
Not

100%|██████████| 27.3M/27.3M [00:00<00:00, 128MB/s] 
Unzipping pnnx-20260526-linux.zip to /content/pnnx-20260526-linux...: 100%|██████████| 3/3 [00:00<00:00,  5.51file/s]

NCNN: running '/usr/local/lib/python3.12/dist-packages/ultralytics/pnnx /content/2026_Hackathon/models/yolo_bear.torchscript ncnnparam=/content/2026_Hackathon/models/yolo_bear_ncnn_model/model.ncnn.param ncnnbin=/content/2026_Hackathon/models/yolo_bear_ncnn_model/model.ncnn.bin ncnnpy=/content/2026_Hackathon/models/yolo_bear_ncnn_model/model_ncnn.py pnnxparam=/content/2026_Hackathon/models/yolo_bear_ncnn_model/model.pnnx.param pnnxbin=/content/2026_Hackathon/models/yolo_bear_ncnn_model/model.pnnx.bin pnnxpy=/content/2026_Hackathon/models/yolo_bear_ncnn_model/model_pnnx.py pnnxonnx=/content/2026_Hackathon/models/yolo_bear_ncnn_model/model.pnnx.onnx fp16=0 device=cpu inputshape="[1, 3, 256, 256]"'


NCNN: export success ✅ 6.8s, saved as '/content/2026_Hackathon/models/yolo_bear_ncnn_model' (11.5 MB)

Export complete (9.4s)
Results saved to /content/2026_Hackathon/models
Predict:         yolo predict task=detect model=/content/2026_Hackathon/models/yolo_bear_ncnn_model imgsz=256  
Validate:        yolo val task=detect model=/content/2026_Hackathon/models/yolo_bear_ncnn_model imgsz=256 data=/content/2026_Hackathon/data/datasets/bear_public_yolo/dataset_colab.yaml  
Visualize:       https://netron.app
ncnn_runtime /content/2026_Hackathon/models/yolo_bear_ncnn_model
exported_model_path /content/2026_Hackathon/models/yolo_bear_ncnn_model


In [12]:
artifact_zip = Path("/content/a1_camera_ai_colab_artifacts.zip")
if artifact_zip.exists():
    artifact_zip.unlink()

include_paths = [
    runtime_pt,
    exported_model_path,
    save_dir / "results.csv",
    save_dir / "args.yaml",
    save_dir / "weights" / "best.pt",
    colab_dataset_yaml,
]

with zipfile.ZipFile(artifact_zip, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for include_path in include_paths:
        include_path = Path(include_path)
        if not include_path.exists():
            continue
        if include_path.is_file():
            archive.write(include_path, include_path.relative_to(workspace) if include_path.is_relative_to(workspace) else include_path.name)
        else:
            for file_path in sorted(include_path.rglob("*")):
                if file_path.is_file():
                    archive.write(file_path, file_path.relative_to(workspace))

print("artifact_zip", artifact_zip, artifact_zip.stat().st_size)
from google.colab import files
files.download(str(artifact_zip))


artifact_zip /content/a1_camera_ai_colab_artifacts.zip 21849424


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>